# 6 · Big data — Datashader & COG viewports

When a layer has millions of points or pixels, sending raw glyphs to the browser is hopeless. The big-data
builders aggregate **server-side** with Datashader and send an image: `rasterize` (a numeric density image,
keeps a Bokeh colorbar + live recolour), `datashade` (RGB, colour-mapped server-side), `trajectory` (shaded
GPS-style tracks) and `large_image` (loads only the visible window of a huge raster/COG at a decimated
overview, via pyramids). We use a 20 000-point synthetic cloud and the global elevation raster.

**Setup** — Bokeh extension, a 20k-point cloud, three random-walk tracks, and the global elevation raster.

In [ ]:
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import holoviews as hv
hv.extension("bokeh")            # the interactive tier renders through Bokeh

import numpy as np, geopandas as gpd, pandas as pd
from pyramids.dataset import Dataset
from digitalearth.interactive import InteractiveMap

rng = np.random.default_rng(0)
N = 20_000
cloud = gpd.GeoDataFrame(
    {"value": rng.normal(10, 2, N)},
    geometry=gpd.points_from_xy(rng.uniform(6.0e5, 9.0e5, N), rng.uniform(6.2e6, 6.5e6, N)),
    crs="EPSG:3857",
)
_frames = []
for tid in ("t1", "t2", "t3"):
    s = rng.normal(0, 4000.0, size=(700, 2)).cumsum(axis=0)
    _frames.append(pd.DataFrame({"x": 7.6e5 + s[:, 0], "y": 6.35e6 + s[:, 1], "track": tid}))
_t = pd.concat(_frames, ignore_index=True)
tracks = gpd.GeoDataFrame(_t[["track"]], geometry=gpd.points_from_xy(_t.x, _t.y), crs="EPSG:3857")

elev = Dataset.read_file(str(DATA / "global" / "wc2.1_10m_elev.tif"))

### `rasterize` — a numeric density image
Aggregate the 20k points to a fixed canvas. We pass `dynamic=False` for a deterministic static image here; in a live server it re-aggregates on zoom and keeps a colorbar.

In [ ]:
m = InteractiveMap(crs=3857, title="20k points → rasterize (count)")
m.rasterize(cloud, aggregator="count", dynamic=False, width=300, height=200)
m

### `datashade` — server-side colour mapping
Like `rasterize` but the colour mapping also happens server-side, so it returns an RGB image (no Bokeh colorbar, but very fast for huge clouds).

In [ ]:
m = InteractiveMap(crs=3857, title="20k points → datashade")
m.datashade(cloud, cmap="fire", dynamic=False, width=300, height=200)
m

### `trajectory` — shaded tracks
Shade many GPS-style tracks, coloured by `track` id, into one RGB layer.

In [ ]:
m = InteractiveMap(crs=3857, title="three tracks (trajectory)")
m.trajectory(tracks, track_column="track", dynamic=False, width=300, height=200)
m

### `large_image` — viewport-loaded COG
Instead of materialising a multi-GB raster, `large_image` reads only the visible window at a suitable overview via pyramids (`preview`/`read_part`). `dynamic=False` renders one decimated preview frame; in a live server it re-reads on pan/zoom. (`band` is 1-based.)

In [ ]:
m = InteractiveMap(crs=elev.epsg, title="global elevation (large_image preview)")
m.large_image(elev, dynamic=False, max_pixels=200 * 200, cmap="terrain")
m